<a href="https://colab.research.google.com/github/sandeepgangaram/neural-nets/blob/master/nanoGPT/nano_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-03-28 02:49:32--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.06s   

2026-03-28 02:49:32 (18.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

1115394

In [4]:
print("length of characters in the dataset:", len(text))

length of characters in the dataset: 1115394


In [6]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [9]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [19]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
encode = lambda s: [stoi[char]for char in s]
decode = lambda l: "".join([itos[ind] for ind in l])
enc = encode("hello world")
decode(enc)


'hello world'

In [22]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
data.shape
# data[:1000]

torch.Size([1115394])

In [25]:
n = int(0.9*len(text))
train_data = data[:n]
val_data = data[n:]

train_data.shape, val_data.shape

(torch.Size([1003854]), torch.Size([111540]))

In [26]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [28]:

for i in range(block_size):
  print(f'For this input {train_data[:i+1]} this is the output {train_data[i+1]}')

For this input tensor([18]) this is the output 47
For this input tensor([18, 47]) this is the output 56
For this input tensor([18, 47, 56]) this is the output 57
For this input tensor([18, 47, 56, 57]) this is the output 58
For this input tensor([18, 47, 56, 57, 58]) this is the output 1
For this input tensor([18, 47, 56, 57, 58,  1]) this is the output 15
For this input tensor([18, 47, 56, 57, 58,  1, 15]) this is the output 47
For this input tensor([18, 47, 56, 57, 58,  1, 15, 47]) this is the output 58


In [40]:
torch.manual_seed(1337)
batch_size = 4 #how many independed sequences will be process in parallel? why for efficiency GPUs are good at parallel processing so why not
block_size = 8 #what is the the maximum context length for predictions?

def get_batch(split):
  data = train_data if split =='train' else val_data
  ix = torch.randint(len(data)-block_size,(batch_size,))

  x = torch.stack([data[i:i+block_size] for i in ix], dim=0)
  y = torch.stack([data[i+1:i+1+block_size] for i in ix], dim=0)

  return x,y

xb,yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for j in range(batch_size):
  for k in range(block_size):
    context = xb[j, :k+1]
    target = yb[j,k]
    print(f"When input is {context.tolist()} the target: {target}")



inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
When input is [24] the target: 43
When input is [24, 43] the target: 58
When input is [24, 43, 58] the target: 5
When input is [24, 43, 58, 5] the target: 57
When input is [24, 43, 58, 5, 57] the target: 1
When input is [24, 43, 58, 5, 57, 1] the target: 46
When input is [24, 43, 58, 5, 57, 1, 46] the target: 43
When input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
When input is [44] the target: 53
When input is [44, 53] the target: 56
When input is [44, 53, 56] the target: 1
When input is [44, 53, 56, 1] the target: 58
When input is [44, 53, 56, 1, 58] the target: 46
When input is [44, 53

In [75]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    #idx and targets are both (B,T) tensor of integers
    logits = self.token_embedding_table(idx) # (B,T,C)

    if targets == None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)

      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_tokens):
    #idx is (B,T) array of indices in the current context
    for _ in range(max_tokens):
      #generate the predictions
      logits, loss = self(idx)
      #focus only on the last time step
      logits = logits[:,-1,:] #becomes (B,C)
      #apply softmax to get probabilities
      probs = F.softmax(logits, dim=-1) #(B,C)
      #sample from the distribution
      idx_next = torch.multinomial(probs, num_samples=1) #(B,1)
      #append sampled index to the running sequence
      idx = torch.cat((idx,idx_next), dim=1) #(B,T+1)
    return idx

m = BigramLanguageModel(vocab_size)
out,loss = m(xb, yb)
print(out.shape, loss)

print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), 100)[0].tolist()))

torch.Size([32, 65]) tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [77]:
#create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [139]:
batch_size = 32

for _ in range(1000):
  #sample a batch of data
  xb,yb = get_batch('train')

  #evaluate the loss
  logits, loss = m(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())
# Note: The original query regarding floating-point comparison (xbow == xbow2) pertains to cell OMnlAhyDJ17C, not this training loop.
# Floating-point numbers can have tiny precision differences, making direct equality (==) unreliable.
# Use torch.allclose() for robust comparisons of floating-point tensors.

2.4747297763824463


In [111]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), 1000)[0].tolist()))


LO:
Noweodicoy ams.
Ayond l Fblior. hy tr lor a t.
Totoult anghait f I y:
Bet

SAMy y:
Famficuchithit ned pew t pale am sershay?-po featad Ans beremeasharny aiokit t f istaler
What, ve peve r gids htary buthandsem p. merate mitheakervofot g.
S mathid th, w berrthar gid t o athru s.
An bif mand tr s thes sout ore Non tunefopathe HAmin velan st; oupll, ll ad th wenatthamy ARELONelmothes hirs a heyon EONGLaut:
TEcermepoug w,
NGELOLin s nd rowerita wicithithte, heconcke m t: athomas llly ou:
Han is,


TARopofus ede tor h'Gawiakior p iore h ty the marorthe hotrmed, mowemupeend the imandlloml mim w lathity he shan br ugayatht?

LE:
TUStingher tligncerour houcene ce deamo usathowonom spu l Be mu forsend.
Soond s; fothterurir d d s ng-sthand:
Se
Thirit my as

AD:
Yowica m ore s h e be cose corin at ughee in amo aves rldd be t gr t saimand
Fo at ck?
AS: ofore:

TELAf chewiserrowoofopes? mfit o II llofest as g be fe ingQNGLAn:

Aseshe bus co'
Sokeer, ougs haramour, f hor lvede he thtus mma ng a

The mathematical trick in self attention


In [129]:
# toy example
torch.manual_seed(1337)
B,T,C = 4,8,2 #batch, time, channels
x = torch.randn(B,T,C)
x.shape


torch.Size([4, 8, 2])

In [130]:
# we want x[b,t] = mean_{i<=1} x[b,i]
xbow = torch.zeros((B,T,C))

for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] # (t,C)
    xbow[b,t] = torch.mean(xprev, 0)
xbow[0],x[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]]))

In [150]:
# the trick - batched matrix multiplication
wei = torch.tril(torch.ones(T,T))
wei = wei/wei.sum(1, keepdims=True)
xbow2 = wei @ x # (B(added), T,T) @(B,T,C) -> (B,T,C)

# Using a small absolute tolerance to account for floating-point differences
# The default rtol is 1e-05 and atol is 1e-08. We'll increase atol slightly.
# print(f"Are xbow and xbow2 close? {torch.allclose(xbow, xbow2, atol=1e-6)}")
# print(xbow[0,5],xbow2[0,5])
# Direct equality will still be false due to tiny precision differences
# print(f"Are xbow[0,5] and xbow2[0,5] exactly equal? {xbow[0,5] == xbow2[0,5]}")
torch.allclose(xbow,xbow2, atol=1e-7)

True

In [165]:
# version 3 using Softmax
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T,T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow,xbow3, atol=1e-7)

True